In [1]:
import pandas as pd
import numpy as np
import sqlite3
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
#Define demand parameters

# ── Demand baseline and patterns ──────────────────────────

# Base demand in megawatt hours (MWh)
BASE_DEMAND = 1000

# Seasonal factors — energy peaks in summer (AC) and winter (heating)
SEASONAL_FACTORS = {
    1:  1.25,   # January   — winter peak
    2:  1.20,   # February  — winter peak
    3:  1.05,   # March     — mild
    4:  0.95,   # April     — mild
    5:  0.90,   # May       — mild
    6:  1.15,   # June      — summer rising
    7:  1.35,   # July      — summer peak
    8:  1.30,   # August    — summer peak
    9:  1.10,   # September — cooling down
    10: 0.95,   # October   — mild
    11: 1.05,   # November  — heating rising
    12: 1.20,   # December  — winter peak
}

# Day of week factors — weekdays higher than weekends
DOW_FACTORS = {
    0: 1.10,    # Monday
    1: 1.12,    # Tuesday
    2: 1.12,    # Wednesday
    3: 1.11,    # Thursday
    4: 1.08,    # Friday
    5: 0.85,    # Saturday
    6: 0.82,    # Sunday
}

# Year over year growth rate
ANNUAL_GROWTH = 0.03   # 3% per year

print("✓ Parameters defined")
print(f"  Base demand: {BASE_DEMAND:,} MWh")
print(f"  Peak month multiplier: {max(SEASONAL_FACTORS.values())}x (July)")
print(f"  Annual growth rate: {ANNUAL_GROWTH*100}%")

✓ Parameters defined
  Base demand: 1,000 MWh
  Peak month multiplier: 1.35x (July)
  Annual growth rate: 3.0%


In [4]:
# ── Generate daily records ─────────────────────────────────

start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 12, 31)

records = []
current = start_date

while current <= end_date:
    # Year-over-year growth factor
    years_elapsed  = (current - start_date).days / 365
    growth_factor  = (1 + ANNUAL_GROWTH) ** years_elapsed

    # Seasonal factor
    seasonal       = SEASONAL_FACTORS[current.month]

    # Day of week factor
    dow            = DOW_FACTORS[current.weekday()]

    # Random noise — normal distribution ±8%
    noise          = np.random.normal(1.0, 0.08)

    # Occasional demand spikes — extreme weather events
    spike          = 1.0
    if random.random() < 0.02:    # 2% chance of spike day
        spike = random.uniform(1.15, 1.35)

    # Calculate final demand
    demand = round(
        BASE_DEMAND * growth_factor * seasonal * dow * noise * spike, 1
    )

    # Temperature simulation — correlated with seasonal demand
    base_temp = {
        1: 35, 2: 38, 3: 50, 4: 62, 5: 72, 6: 82,
        7: 90, 8: 88, 9: 78, 10: 64, 11: 50, 12: 38
    }[current.month]
    temperature = round(base_temp + np.random.normal(0, 5), 1)

    records.append({
        "date":        current.strftime("%Y-%m-%d"),
        "year":        current.year,
        "month":       current.month,
        "quarter":     (current.month - 1) // 3 + 1,
        "day_of_week": current.strftime("%A"),
        "is_weekend":  1 if current.weekday() >= 5 else 0,
        "season":      (
            "Winter" if current.month in [12, 1, 2] else
            "Spring" if current.month in [3, 4, 5]  else
            "Summer" if current.month in [6, 7, 8]  else
            "Fall"
        ),
        "temperature_f": temperature,
        "demand_mwh":    demand,
        "is_spike":      1 if spike > 1.0 else 0,
    })

    current += timedelta(days=1)

df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"])

print(f"✓ Dataset generated: {len(df):,} rows")
print(f"✓ Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"✓ Avg daily demand: {df['demand_mwh'].mean():,.1f} MWh")
print(f"✓ Peak demand: {df['demand_mwh'].max():,.1f} MWh")
print(f"✓ Min demand: {df['demand_mwh'].min():,.1f} MWh")
print(f"✓ Spike days: {df['is_spike'].sum()}")
print(f"\nAvg demand by season:")
print(df.groupby("season")["demand_mwh"].mean().round(1).sort_values(ascending=False))

✓ Dataset generated: 1,096 rows
✓ Date range: 2022-01-01 to 2024-12-31
✓ Avg daily demand: 1,211.9 MWh
✓ Peak demand: 2,170.3 MWh
✓ Min demand: 643.6 MWh
✓ Spike days: 19

Avg demand by season:
season
Summer    1373.0
Winter    1304.8
Fall      1123.6
Spring    1047.1
Name: demand_mwh, dtype: float64


In [6]:
#Add monthly aggregates
# ── Monthly summary table ──────────────────────────────────
# This is what we'll use for forecasting — daily is too noisy

monthly = (
    df.groupby(["year", "month"])
    .agg(
        total_demand   = ("demand_mwh",    "sum"),
        avg_daily      = ("demand_mwh",    "mean"),
        peak_demand    = ("demand_mwh",    "max"),
        min_demand     = ("demand_mwh",    "min"),
        avg_temp       = ("temperature_f", "mean"),
        spike_days     = ("is_spike",      "sum"),
        days_in_month  = ("demand_mwh",    "count")
    )
    .reset_index()
)

monthly["year_month"] = (
    monthly["year"].astype(str) + "-" +
    monthly["month"].astype(str).str.zfill(2)
)

monthly = monthly.round(1)

print(f"✓ Monthly summary: {len(monthly)} rows")
print(f"\nSample monthly data:")
print(monthly[["year_month", "total_demand", "avg_daily",
               "avg_temp"]].head(6).to_string(index=False))

✓ Monthly summary: 36 rows

Sample monthly data:
year_month  total_demand  avg_daily  avg_temp
   2022-01       39500.8     1274.2      34.2
   2022-02       34307.2     1225.3      38.4
   2022-03       33894.8     1093.4      51.4
   2022-04       29682.8      989.4      62.6
   2022-05       29738.7      959.3      71.0
   2022-06       36689.3     1223.0      82.1


In [7]:
#Save to SQLite and CSV

# ── Save to SQLite ─────────────────────────────────────────
conn = sqlite3.connect("../data/energy.db")
df.to_sql("daily_demand",   conn, if_exists="replace", index=False)
monthly.to_sql("monthly_demand", conn, if_exists="replace", index=False)
conn.close()
print("✓ Database saved to data/energy.db")

# ── Save to CSV ────────────────────────────────────────────
df.to_csv("../data/daily_demand.csv",   index=False)
monthly.to_csv("../data/monthly_demand.csv", index=False)
print("✓ CSVs saved to data/")

# ── Sanity check ───────────────────────────────────────────
conn = sqlite3.connect("../data/energy.db")
print("\n── Tables in database ──────────────────────────────")
print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
).to_string(index=False))

print("\n── Annual demand summary ───────────────────────────")
print(pd.read_sql("""
    SELECT year,
           COUNT(*)                        AS days,
           ROUND(SUM(demand_mwh), 1)       AS total_demand_mwh,
           ROUND(AVG(demand_mwh), 1)       AS avg_daily_mwh,
           ROUND(MAX(demand_mwh), 1)       AS peak_day_mwh
    FROM daily_demand
    GROUP BY year
    ORDER BY year
""", conn).to_string(index=False))
conn.close()

✓ Database saved to data/energy.db
✓ CSVs saved to data/

── Tables in database ──────────────────────────────
          name
  daily_demand
monthly_demand

── Annual demand summary ───────────────────────────
 year  days  total_demand_mwh  avg_daily_mwh  peak_day_mwh
 2022   365          427690.2         1171.8        1783.4
 2023   365          446849.9         1224.2        2043.9
 2024   366          453732.4         1239.7        2170.3
